[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/templates/48_tta_hflip.ipynb)

# 🟢 Easy: TTA (Horizontal Flip Averaging)

**Test-Time Augmentation** with horizontal flip を実装する。model を `x` と `x.flip(-1)` の両方で走らせ、softmax 確率を **平均**（logit じゃない）。image classification で 0.3〜1.0% タダで上がる。

> 💡 **どこで使う**：推論時の augmentation。test 画像と flip 版で 2 回 forward、softmax 確率を平均。

<details>
<summary>📖 もっと詳しく</summary>

0.3–1% タダで accuracy が上がる、Kaggle の鉄板。

Probability-space (softmax 後) で平均するのが標準 — logit-space (softmax 前) より自然な ensemble に。5-crop (center + 4 corner) との組み合わせも可能、推論 cost は 2x-10x になるが多くの場合元が取れる。

</details>

### 標準作法 — probability-space averaging
softmax の **後**で平均する（logit を平均してから softmax するんじゃない）。各 TTA 分岐を独立 classifier と見て predictions を ensemble する流儀。Kaggle / 論文の慣例。


### Signature
```python
def tta_hflip(model, x):
    # model: nn.Module taking (B, C, H, W) → (B, num_classes) logits
    # x: (B, C, H, W) input
    # Returns: (B, num_classes) averaged probabilities
```

### Rules
- softmax を **先に**適用してから平均（probability-space ensemble、logit-space ではない）
- `torch.no_grad()` で wrap — 推論なので autograd tape 不要
- *確率* (各行 sum=1) を return、logit ではない
- model state を変更しない（caller が `model.eval()` 済みと仮定）

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q --force-reinstall --no-deps git+https://github.com/alextfkd/TorchCode.git')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def tta_hflip(model, x):
    pass  # softmax(model(x)) + softmax(model(x.flip(-1))) / 2、all under no_grad

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)

model = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Linear(8, 10),
)
model.eval()
x = torch.randn(2, 3, 16, 16)
p = tta_hflip(model, x)
print('Output shape:', p.shape, '| row sums:', p.sum(dim=-1).tolist())

In [ ]:
# ✅ SUBMIT — Run this cell to check your solution
from torch_judge import check
check("tta_hflip")

---

**詰まったら？** 解答を見る：

[![Open Solution in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alextfkd/TorchCode/blob/master/solutions/48_tta_hflip_solution.ipynb) または [GitHub で開く](https://github.com/alextfkd/TorchCode/blob/master/solutions/48_tta_hflip_solution.ipynb)